# HKUST Smart Meter Dataset — Eksplorasi & Pemilihan Subset
**Tujuan:** Memahami struktur data, memetakan meter ke gedung via TTL, dan memilih subset terbaik untuk analisis.

**Ganti path file di cell berikut sesuai lokasi file kamu.**

In [ ]:
# =============================================
# INSTALL DEPENDENCIES (jalankan sekali saja)
# =============================================
# !pip install rdflib openpyxl pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdflib import Graph, Namespace, RDF, RDFS
import warnings
warnings.filterwarnings('ignore')

# =============================================
# GANTI PATH INI SESUAI LOKASI FILE KAMU
# =============================================
EXCEL_PATH = 'data/t1440.xlsx'   # path ke file Excel t1440
TTL_PATH   = 'data/metadata.ttl' # path ke file .ttl

print('Libraries loaded OK')

## STEP 1 — Baca & Pahami Struktur Excel

In [ ]:
# Baca Excel
df = pd.read_excel(EXCEL_PATH, index_col=0, parse_dates=True)

print('=== INFO UMUM ===')
print(f'Jumlah baris (hari)   : {df.shape[0]}')
print(f'Jumlah kolom (meter)  : {df.shape[1]}')
print(f'Periode awal          : {df.index.min()}')
print(f'Periode akhir         : {df.index.max()}')
print()
print('5 baris pertama:')
df.head()

In [ ]:
# Cek nama kolom (meter IDs)
print('Contoh 20 meter ID pertama:')
print(list(df.columns[:20]))
print()
print('Contoh 20 meter ID terakhir:')
print(list(df.columns[-20:]))

## STEP 2 — Analisis Kelengkapan Data per Meter
Pilih meter yang datanya **paling lengkap** (missing value rendah)

In [ ]:
# Hitung persentase missing value per meter
total_rows = len(df)
missing_pct = df.isnull().sum() / total_rows * 100
completeness = 100 - missing_pct

completeness_df = pd.DataFrame({
    'meter_id'      : df.columns,
    'missing_pct'   : missing_pct.values,
    'completeness'  : completeness.values,
    'mean_kwh'      : df.mean().values,
    'max_kwh'       : df.max().values,
    'std_kwh'       : df.std().values,
}).reset_index(drop=True)

print(f'Total meter: {len(completeness_df)}')
print(f'Meter dengan completeness >= 90%: {(completeness_df.completeness >= 90).sum()}')
print(f'Meter dengan completeness >= 80%: {(completeness_df.completeness >= 80).sum()}')
completeness_df.sort_values('completeness', ascending=False).head(20)

In [ ]:
# Visualisasi distribusi kelengkapan data
plt.figure(figsize=(10, 4))
plt.hist(completeness_df['completeness'], bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Completeness (%)')
plt.ylabel('Jumlah Meter')
plt.title('Distribusi Kelengkapan Data per Meter')
plt.axvline(90, color='red', linestyle='--', label='90% threshold')
plt.legend()
plt.tight_layout()
plt.show()

## STEP 3 — Parse TTL Metadata
Petakan meter ID → nama gedung/zona

In [ ]:
# Parse TTL dengan rdflib
g = Graph()
g.parse(TTL_PATH, format='turtle')
print(f'TTL berhasil dibaca. Jumlah triple: {len(g)}')

In [ ]:
# Ekstrak relasi meter → label/nama
# Coba beberapa predicate umum di Brick Schema
results = []

for s, p, o in g:
    s_str = str(s)
    p_str = str(p)
    o_str = str(o)
    results.append({'subject': s_str, 'predicate': p_str, 'object': o_str})

rdf_df = pd.DataFrame(results)
print(f'Total triple: {len(rdf_df)}')
print('\nContoh predicate yang ada:')
print(rdf_df['predicate'].value_counts().head(20))

In [ ]:
# Cari label/nama untuk setiap entitas
label_predicates = [
    'http://www.w3.org/2000/01/rdf-schema#label',
    'http://www.w3.org/2004/02/skos/core#prefLabel',
]

labels = rdf_df[rdf_df['predicate'].isin(label_predicates)][['subject', 'object']]
labels.columns = ['entity', 'label']
print(f'Entitas dengan label: {len(labels)}')
labels.head(20)

In [ ]:
# Cari relasi isPartOf / hasLocation / isLocationOf untuk meter → gedung
location_predicates = [
    'https://brickschema.org/schema/Brick#isPartOf',
    'https://brickschema.org/schema/Brick#hasLocation',
    'https://brickschema.org/schema/Brick#isLocationOf',
    'https://brickschema.org/schema/Brick#feeds',
    'https://brickschema.org/schema/Brick#isFedBy',
]

loc_df = rdf_df[rdf_df['predicate'].isin(location_predicates)]
print(f'Relasi lokasi ditemukan: {len(loc_df)}')
loc_df.head(20)

In [ ]:
# Cek semua predicate yang mengandung kata 'part', 'location', 'building', 'zone'
keywords = ['part', 'location', 'building', 'zone', 'floor', 'room', 'feeds']
mask = rdf_df['predicate'].str.lower().apply(
    lambda x: any(k in x for k in keywords)
)
print('Predicate yang relevan:')
print(rdf_df[mask]['predicate'].value_counts())

In [ ]:
# Lihat semua subject yang mengandung nama meter dari Excel
# Ambil beberapa meter ID dari Excel untuk cek apakah ada di TTL
sample_meters = list(df.columns[:5])
print('Sample meter IDs dari Excel:', sample_meters)
print()

for m in sample_meters:
    matches = rdf_df[rdf_df['subject'].str.contains(str(m), na=False)]
    if len(matches) > 0:
        print(f'=== Meter: {m} ===')
        print(matches[['predicate', 'object']].to_string())
        print()

## STEP 4 — Buat Mapping Meter → Gedung
(Sesuaikan query SPARQL di bawah berdasarkan hasil STEP 3)

In [ ]:
# SPARQL query untuk mapping meter → gedung
# Sesuaikan namespace berdasarkan hasil step 3
query = """
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?meter ?meter_label ?location ?location_label
WHERE {
    ?meter a brick:Meter .
    OPTIONAL { ?meter rdfs:label ?meter_label }
    OPTIONAL {
        ?meter brick:isPartOf ?location .
        OPTIONAL { ?location rdfs:label ?location_label }
    }
}
"""

mapping_results = []
for row in g.query(query):
    mapping_results.append({
        'meter'          : str(row.meter).split('#')[-1].split('/')[-1],
        'meter_label'    : str(row.meter_label) if row.meter_label else '',
        'location'       : str(row.location).split('#')[-1].split('/')[-1] if row.location else '',
        'location_label' : str(row.location_label) if row.location_label else '',
    })

mapping_df = pd.DataFrame(mapping_results)
print(f'Meter terpetakan: {len(mapping_df)}')
mapping_df.head(20)

## STEP 5 — Gabungkan Metadata + Completeness
Pilih meter terbaik berdasarkan kelengkapan data + keragaman gedung

In [ ]:
# Gabungkan completeness_df dengan mapping_df
merged = completeness_df.merge(
    mapping_df[['meter', 'meter_label', 'location', 'location_label']],
    left_on='meter_id', right_on='meter',
    how='left'
).drop(columns=['meter'])

# Filter: completeness >= 80%, ada lokasi
good_meters = merged[
    (merged['completeness'] >= 80) &
    (merged['location'] != '')
].sort_values('completeness', ascending=False)

print(f'Meter berkualitas baik dengan lokasi: {len(good_meters)}')
good_meters.head(30)

In [ ]:
# Lihat daftar gedung yang tersedia
print('Gedung/zona yang tersedia:')
print(good_meters['location_label'].value_counts())

In [ ]:
# =============================================
# PILIH GEDUNG yang ingin dianalisis
# Ganti nama gedung di sini setelah melihat output cell atas
# =============================================
SELECTED_BUILDINGS = [
    # Contoh — ganti dengan nama gedung dari output di atas:
    # 'Academic Building',
    # 'Library',
    # 'Student Residence',
    # 'Lab Building',
]

if SELECTED_BUILDINGS:
    selected_meters_df = good_meters[
        good_meters['location_label'].isin(SELECTED_BUILDINGS)
    ]
else:
    # Jika belum tahu nama gedung, ambil top N meter by completeness
    # ambil 1 meter terbaik per lokasi
    selected_meters_df = good_meters.groupby('location_label').apply(
        lambda x: x.nlargest(1, 'completeness')
    ).reset_index(drop=True).head(10)

print(f'Meter terpilih: {len(selected_meters_df)}')
selected_meters_df[['meter_id','location','location_label','completeness','mean_kwh']]

## STEP 6 — Buat Dataset Final (Subset)
Simpan sebagai CSV untuk digunakan di notebook analisis selanjutnya

In [ ]:
# Ambil kolom meter yang terpilih
selected_meter_ids = selected_meters_df['meter_id'].tolist()

# Filter kolom dari Excel
available = [m for m in selected_meter_ids if m in df.columns]
df_subset = df[available].copy()

print(f'Meter tersedia di Excel: {len(available)} dari {len(selected_meter_ids)}')
print(f'Shape subset: {df_subset.shape}')
df_subset.head()

In [ ]:
# Rename kolom agar lebih informatif (meter_id → nama gedung)
rename_map = dict(zip(
    selected_meters_df['meter_id'],
    selected_meters_df['location_label'] + '_' + selected_meters_df['meter_id']
))
df_subset.rename(columns=rename_map, inplace=True)

# Simpan
df_subset.to_csv('data/clean/subset_t1440.csv')
print('Dataset subset disimpan ke data/clean/subset_t1440.csv')
df_subset.head()

In [ ]:
# Simpan juga metadata meter terpilih
selected_meters_df.to_csv('data/clean/selected_meters_metadata.csv', index=False)
print('Metadata meter terpilih disimpan ke data/clean/selected_meters_metadata.csv')

## STEP 7 — Preview Cepat Data Terpilih

In [ ]:
# Plot konsumsi harian semua meter terpilih
fig, ax = plt.subplots(figsize=(14, 5))
df_subset.plot(ax=ax, alpha=0.7, linewidth=1)
ax.set_title('Konsumsi Energi Harian — Meter Terpilih')
ax.set_xlabel('Tanggal')
ax.set_ylabel('Konsumsi (kWh)')
ax.legend(bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Total konsumsi per meter (bar chart)
total_kwh = df_subset.sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
total_kwh.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Total Konsumsi Energi per Meter/Gedung (Seluruh Periode)')
ax.set_xlabel('Meter / Gedung')
ax.set_ylabel('Total kWh')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nRingkasan:')
print(total_kwh)

## Catatan untuk Laporan Bab 2 (Obtain)

Gunakan output notebook ini untuk mengisi tabel dokumentasi dataset:

| No | Nama Dataset | Sumber | Periode | Format | Baris | Kolom | Peran |
|---|---|---|---|---|---|---|---|
| 1 | HKUST Smart Meter (t1440) | Dryad | 2022-01 s/d 2024-05 | .xlsx | [isi] | [isi] | Utama |
| 2 | Building Metadata | Dryad (.ttl) | — | .ttl | — | — | Pendukung |

**Alasan pemilihan subset:**
- Interval t1440 dipilih karena ukuran data lebih ringan dan cukup untuk analisis tren harian, mingguan, dan bulanan
- Meter dipilih berdasarkan kelengkapan data (completeness ≥ 80%) dan keragaman fungsi gedung
- Jumlah meter terpilih: [isi setelah menjalankan notebook]